In [ ]:
import socket
import numpy as np

class DataStream:
    def __init__(self, IP, PORT, 
                 buffer_size=1024,
                 total_channels=8,      # Total number of hardware channels
                 used_channels=7,       # Number of actually used channels (take first N)
                 pkg_groups=1,          # Number of time points per network packet
                 data_group_len=250):   # Number of time points returned per __next__ call
        """
        Online data stream: receive neural signal data in CSV format from TCP server.
        
        Data format assumption (per packet):
            timestamp, marker, ch0_t0, ch1_t0, ..., ch7_t0, ch0_t1, ..., ch7_t4
            -> Total 2 + pkg_groups * total_channels fields
        """
        self.ip = IP
        self.port = PORT
        self.buffer_size = buffer_size
        self.total_channels = total_channels
        self.used_channels = used_channels
        self.pkg_groups = pkg_groups
        self.data_group_len = data_group_len
        
        self.is_running = False
        self.socket = None
        self._buffer_str = ""      # Accumulated unparsed raw string
        self._data_buffer = []     # Accumulated parsed time points (each with shape=(used_channels,))

    def __iter__(self):
        if self.is_running:
            self.close()
        self.is_running = True
        self._connect()
        # Reset buffer
        self._buffer_str = ""
        self._data_buffer = []
        return self

    def _connect(self):
        try:
            self.socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.socket.settimeout(5)
            self.socket.connect((self.ip, self.port))
            self.socket.send(b'start')
            print(f"[DataStream] Connected to {self.ip}:{self.port}")
        except Exception as e:
            print(f"[DataStream] Connection Error: {e}")
            self.is_running = False
            raise

    def close(self):
        self.is_running = False
        if self.socket:
            try:
                self.socket.close()
            except:
                pass
            self.socket = None

    def __next__(self):
        if not self.is_running:
            raise StopIteration

        while len(self._data_buffer) < self.data_group_len:
            try:
                chunk = self.socket.recv(self.buffer_size)
                if not chunk:
                    raise ConnectionError("Server closed connection.")
                self._buffer_str += chunk.decode('utf-8', errors='ignore')

                # Try to parse all complete data packets
                while True:
                    lines = self._buffer_str.split('\n')
                    if len(lines) < 2:
                        break  # No complete line
                    # Process all complete lines (except the last which may be incomplete)
                    complete_lines = lines[:-1]
                    self._buffer_str = lines[-1]  # Keep incomplete tail

                    for line in complete_lines:
                        if not line.strip():
                            continue
                        fields = line.strip().split(',')
                        expected_fields = 2 + self.pkg_groups * self.total_channels
                        if len(fields) < expected_fields:
                            continue  # Incomplete data, skip

                        # Parse data part: skip first 2 fields (timestamp, marker)
                        try:
                            data_vals = list(map(float, fields[2:2 + self.pkg_groups * self.total_channels]))
                        except ValueError:
                            continue  # Conversion failed, skip

                        # Convert to (pkg_groups, total_channels) -> take first used_channels
                        arr = np.array(data_vals, dtype=np.float32)
                        arr = arr.reshape(self.pkg_groups, self.total_channels)
                        arr = arr[:, :self.used_channels]  # shape: (pkg_groups, used_channels)

                        # Add each time point to buffer (each time point is (used_channels,))
                        for t in range(self.pkg_groups):
                            self._data_buffer.append(arr[t].tolist())

                    # Check if enough data has been collected
                    if len(self._data_buffer) >= self.data_group_len:
                        break

            except socket.timeout:
                continue
            except Exception as e:
                print(f"[DataStream] Receive/Parse Error: {e}")
                self.close()
                raise StopIteration

        # Return required amount of data (list of lists)
        result = self._data_buffer[:self.data_group_len]
        self._data_buffer = self._data_buffer[self.data_group_len:]  # Keep excess data
        return result

    def __del__(self):
        self.close()

In [ ]:
stream = DataStream(
    IP="127.0.0.1",
    PORT=9600,
    total_channels=8,
    used_channels=7,
    pkg_groups=1,
    data_group_len=250
)

try:
    for data_group in stream:  # Returns 250 time points x 7 channels each time
        arr = np.array(data_group)  # shape: (250, 7)
        print("Received:", arr.shape)
        # Process data here...
except KeyboardInterrupt:
    pass
finally:
    stream.close()